In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from config.paths import LABELS_ROOT, OUTPUT_ROOT

labels_root = LABELS_ROOT
graphs_folder = "label_statistics_plots"
graphs_root = OUTPUT_ROOT / graphs_folder
graphs_root.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.figsize": (13, 7),
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 150,
})

VIOLIN_POINTS = 50
VIOLIN_BW_METHOD = 0.15


def discover_processed_missions(labels_root: Path):
    """
    Discover all processed missions by scanning labels_root for <mission>/rhe
    sub-directories that contain the expected .npy output files.
    Returns a list of (mission_short_name, rhe_dir, simple_dir) tuples.
    """
    processed = []
    if not labels_root.exists():
        return processed

    for subdir in labels_root.iterdir():
        if not subdir.is_dir():
            continue
        rhe_dir = subdir / "rhe"
        # Check if this directory has the expected outputs
        if (rhe_dir / "rhe_trav_labels.npy").exists() and (
            rhe_dir / "rhe_timestamp.npy"
        ).exists():
            processed.append((subdir.name, rhe_dir, subdir / "simple"))

    return sorted(processed)


def load_mission_statistics(output_dir: Path, mission_short_name: str):
    timestamp_path = output_dir / "rhe_timestamp.npy"
    labels_path = output_dir / "rhe_trav_labels.npy"
    mu_conf_path = output_dir / "rhe_mu_confidences.npy"
    nu_conf_path = output_dir / "rhe_nu_confidences.npy"

    if not timestamp_path.exists() or not labels_path.exists():
        print(
            f"Skipping {mission_short_name}: missing analysis outputs in {output_dir}"
        )
        return None

    trav_labels = np.load(labels_path)
    mu_confidences = np.load(mu_conf_path) if mu_conf_path.exists() else np.array([])
    nu_confidences = np.load(nu_conf_path) if nu_conf_path.exists() else np.array([])

    mu_labels = trav_labels[:, 0]
    nu_labels = trav_labels[:, 1]

    high_conf_mu = (
        mu_labels[mu_confidences >= CONFIDENCE_THRESHOLD]
        if mu_confidences.size
        else np.array([])
    )
    high_conf_nu = (
        nu_labels[nu_confidences >= CONFIDENCE_THRESHOLD]
        if nu_confidences.size
        else np.array([])
    )
    combined_confidences = (
        np.concatenate([arr for arr in [mu_confidences, nu_confidences] if arr.size])
        if (mu_confidences.size or nu_confidences.size)
        else np.array([])
    )

    return {
        "mission_short_name": mission_short_name,
        "mu_labels": mu_labels,
        "nu_labels": nu_labels,
        "high_conf_mu": high_conf_mu,
        "high_conf_nu": high_conf_nu,
        "mu_confidences": mu_confidences,
        "nu_confidences": nu_confidences,
        "combined_confidences": combined_confidences,
    }


def load_simple_mission_statistics(output_dir: Path, mission_short_name: str):
    """Load simple baseline label outputs produced by SimpleBaselineLabeler."""
    tau_lin_path = output_dir / "simple_trav_lin.npy"
    tau_ang_path = output_dir / "simple_trav_ang.npy"

    if not tau_lin_path.exists() or not tau_ang_path.exists():
        print(
            f"Skipping {mission_short_name}: missing simple label outputs in {output_dir}"
        )
        return None

    tau_lin = np.load(tau_lin_path)
    tau_ang = np.load(tau_ang_path)

    return {
        "mission_short_name": mission_short_name,
        "tau_lin": tau_lin,
        "tau_ang": tau_ang,
    }


def concatenate_non_empty(arrays):
    non_empty = [array for array in arrays if array.size]
    if not non_empty:
        return np.array([])
    return np.concatenate(non_empty)


def save_figure(fig, filename: str):
    path = graphs_root / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {path}")


def add_mean_median_vlines(ax, values, ymin, ymax, include_in_legend: bool = True):
    if not values.size:
        return

    # mean_label = f"mean={np.mean(values):.3f}" if include_in_legend else "_nolegend_"
    mean_label = "mean" if include_in_legend else "_nolegend_"
    median_label = "median" if include_in_legend else "_nolegend_"

    ax.vlines(
        float(np.mean(values)),
        ymin,
        ymax,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=mean_label,
    )
    ax.vlines(
        float(np.median(values)),
        ymin,
        ymax,
        color="tab:orange",
        linestyle=":",
        linewidth=1.8,
        label=median_label,
    )


def render_hist_panel(ax, values, bins, color, xlabel, panel_title):
    if values.size:
        ax.hist(
            values, bins=bins, density=True, color=color, alpha=0.75, edgecolor="white"
        )
        add_mean_median_vlines(ax, values, 0.0, ax.get_ylim()[1])
        ax.legend(loc="best")
    else:
        ax.text(
            0.5,
            0.5,
            "No values available",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

    ax.set_title(panel_title)
    ax.set_xlabel(xlabel)
    ax.set_xlim(0.0, 1.0)
    ax.grid(True, alpha=0.3)


def plot_distribution(values, title: str, xlabel: str, filename: str, color: str):
    if not values.size:
        print(f"Skipping {filename}: no values available.")
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    bins = np.linspace(0.0, 1.0, 41)
    render_hist_panel(ax, values, bins, color, xlabel, title)
    ax.set_ylabel("Density")
    save_figure(fig, filename)


def plot_confidence_distribution(values, filename: str):
    plot_distribution(
        values,
        title="Confidence Distribution Across the Entire Dataset",
        xlabel="Confidence",
        filename=filename,
        color="tab:purple",
    )


def histogram_density(values, bins, density_dist=True):
    if not values.size:
        return None, None
    density_dist, edges = np.histogram(values, bins=bins, density=density_dist)
    centers = 0.5 * (edges[1:] + edges[:-1])
    return centers, density_dist


def draw_above_below_on_axis(
    ax,
    top_values,
    bottom_values,
    bins,
    top_color,
    bottom_color,
    top_label=None,
    bottom_label=None,
    show_legend: bool = True,
    density_dist=True,
):
    top_max = 0.0
    bottom_max = 0.0

    centers, density = histogram_density(top_values, bins, density_dist=density_dist)
    if centers is not None:
        ax.fill_between(
            centers,
            0,
            density,
            step="mid",
            color=top_color,
            alpha=0.75,
            label=top_label,
        )
        top_max = float(np.max(density))
        add_mean_median_vlines(ax, top_values, 0.0, top_max, include_in_legend=True)

    centers, density = histogram_density(bottom_values, bins, density_dist=density_dist)
    if centers is not None:
        ax.fill_between(
            centers,
            0,
            -density,
            step="mid",
            color=bottom_color,
            alpha=0.75,
            label=bottom_label,
        )
        bottom_max = float(np.max(density))
        add_mean_median_vlines(
            ax, bottom_values, -bottom_max, 0.0, include_in_legend=False
        )

    max_density = max(top_max, bottom_max)
    if max_density > 0:
        ax.set_ylim(-max_density * 1.1, max_density * 1.1)

    ax.axhline(0.0, color="black", linewidth=1.0)
    ax.grid(True, alpha=0.3)
    if show_legend and (top_label or bottom_label):
        ax.legend(loc="best")


def plot_above_below_distribution(
    top_values,
    bottom_values,
    filename: str,
    title: str,
    xlabel: str,
    top_label: str,
    bottom_label: str,
    top_color: str = "tab:blue",
    bottom_color: str = "tab:red",
):
    if not top_values.size and not bottom_values.size:
        print(f"Skipping {filename}: no values available.")
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    bins = np.linspace(0.0, 1.0, 41)
    draw_above_below_on_axis(
        ax,
        top_values,
        bottom_values,
        bins,
        top_color,
        bottom_color,
        top_label=top_label,
        bottom_label=bottom_label,
        show_legend=True,
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_xlim(0.0, 1.0)
    save_figure(fig, filename)


def plot_linear_angular_confidence_distributions(
    mu_values,
    nu_values,
    filename: str,
    split_into_files: bool = False,
):
    if split_into_files:
        plot_distribution(
            mu_values,
            title="Linear Confidence Distribution Across the Entire Dataset",
            xlabel="Linear confidence (mu)",
            filename="linear_confidence_distribution_all_missions.pdf",
            color="tab:blue",
        )
        plot_distribution(
            nu_values,
            title="Angular Confidence Distribution Across the Entire Dataset",
            xlabel="Angular confidence (nu)",
            filename="angular_confidence_distribution_all_missions.pdf",
            color="tab:red",
        )
        return

    plot_above_below_distribution(
        mu_values,
        nu_values,
        filename=filename,
        title="Confidence Distributions Across the Entire Dataset",
        xlabel="Confidence",
        top_label="Linear confidence (mu)",
        bottom_label="Angular confidence (nu)",
    )


def plot_high_confidence_traversability_distributions(
    mu_values,
    nu_values,
    filename: str,
    split_into_files: bool = False,
):
    if split_into_files:
        plot_distribution(
            mu_values,
            title=f"Linear Traversability Distribution for Confidence >= {CONFIDENCE_THRESHOLD:.1f}",
            xlabel="Linear traversability coefficient (mu)",
            filename=f"linear_traversability_high_confidence_distribution_{CONFIDENCE_THRESHOLD:.1f}.pdf",
            color="tab:blue",
        )
        plot_distribution(
            nu_values,
            title=f"Angular Traversability Distribution for Confidence >= {CONFIDENCE_THRESHOLD:.1f}",
            xlabel="Angular traversability coefficient (nu)",
            filename=f"angular_traversability_high_confidence_distribution_{CONFIDENCE_THRESHOLD:.1f}.pdf",
            color="tab:red",
        )
        return

    plot_above_below_distribution(
        mu_values,
        nu_values,
        filename=filename,
        title=f"Traversability Distributions for Confidence >= {CONFIDENCE_THRESHOLD:.1f}",
        xlabel="Traversability coefficient",
        top_label="Linear traversability (mu)",
        bottom_label="Angular traversability (nu)",
    )


def mission_low_traversability_count(
    stats,
    low_value_threshold: float,
    confidence_threshold: float,
):
    mu_labels = np.asarray(stats.get("mu_labels", np.array([])))
    nu_labels = np.asarray(stats.get("nu_labels", np.array([])))
    mu_conf = np.asarray(stats.get("mu_confidences", np.array([])))
    nu_conf = np.asarray(stats.get("nu_confidences", np.array([])))

    if mu_labels.size and mu_conf.size:
        mu_mask = mu_conf > confidence_threshold
        mu_count = int(np.sum(mu_labels[mu_mask] < low_value_threshold))
    else:
        mu_count = 0

    if nu_labels.size and nu_conf.size:
        nu_mask = nu_conf > confidence_threshold
        nu_count = int(np.sum(nu_labels[nu_mask] < low_value_threshold))
    else:
        nu_count = 0

    return mu_count + nu_count


def filter_low_traversability_missions(
    stats_list,
    low_value_threshold: float,
    min_count: int,
    confidence_threshold: float,
):
    return [
        stats
        for stats in stats_list
        if mission_low_traversability_count(
            stats,
            low_value_threshold,
            confidence_threshold,
        )
        > min_count
    ]


def plot_mission_traversability_distributions(
    stats_list,
    filename: str,
    low_value_threshold: float,
    min_count: int,
    confidence_threshold: float,
):
    filtered = filter_low_traversability_missions(
        stats_list,
        low_value_threshold,
        min_count,
        confidence_threshold,
    )
    if not filtered:
        print(
            "Skipping filtered traversability plot: no missions meet the low traversability criteria."
        )
        return

    bins = np.linspace(0.0, 1.0, 41)
    fig, axes = plt.subplots(
        len(filtered),
        1,
        figsize=(12, max(3, len(filtered) * 2)),
        sharex=True,
        sharey=False,
    )

    if len(filtered) == 1:
        axes = np.array([axes])

    for row_idx, stats in enumerate(filtered):
        name = stats["mission_short_name"]
        mu = stats["mu_labels"]  # [stats["mu_labels"] < low_value_threshold]
        nu = stats["nu_labels"]  # [stats["nu_labels"] < low_value_threshold]
        ax = axes[row_idx]

        if mu.size == 0 and nu.size == 0:
            ax.text(
                0.5,
                0.5,
                "No traversability data",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_ylabel(name)
            continue

        draw_above_below_on_axis(
            ax,
            mu,
            nu,
            bins,
            "tab:blue",
            "tab:red",
            top_label="Linear traversability (mu)" if row_idx == 0 else None,
            bottom_label="Angular traversability (nu)" if row_idx == 0 else None,
            show_legend=row_idx == 0,
            density_dist=False,
        )

        # Draw the threshold line
        ax.axvline(
            low_value_threshold,
            color="black",
            linestyle="-",
            linewidth=1.5,
            label=f"Low traversability threshold ({low_value_threshold:.2f})",
        )

        ax.set_ylabel(name)
        ax.set_xlim(0.0, 1.0)

        def max_hist_density_below_threshold(values, threshold):
            if not values.size:
                return 0.0
            centers, density = histogram_density(values, bins, density_dist=False)
            if centers is None or density is None or not density.size:
                return 0.0
            below_threshold = centers < threshold
            if not np.any(below_threshold):
                return 0.0
            return float(np.max(density[below_threshold]))

        ylim_low_density = max_hist_density_below_threshold(nu, low_value_threshold)
        ylim_high_density = max_hist_density_below_threshold(mu, low_value_threshold)

        lim = max(ylim_low_density, ylim_high_density)
        ax.set_ylim(-lim * 1.1, lim * 1.1)
    axes[-1].set_xlabel("Traversability coefficient")
    fig.supylabel("Density")
    fig.suptitle(
        "Traversability Distributions for Missions with at least "
        f"{min_count} Low Traversability Windows (Threshold < {low_value_threshold:.2f}, "
        f"Confidence > {confidence_threshold:.2f})",
        y=1.02,
    )
    fig.tight_layout()
    save_figure(fig, filename)


def plot_high_confidence_traversability_log_distributions(
    mu_values,
    nu_values,
    filename: str,
    confidence_threshold: float,
):
    if not mu_values.size and not nu_values.size:
        print(
            "Skipping log-scale traversability plot: no high-confidence values available."
        )
        return

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, sharey=False)
    bins = np.linspace(0.0, 1.0, 41)

    panels = [
        (axes[0], mu_values, "tab:blue", "Linear traversability (mu)"),
        (axes[1], nu_values, "tab:red", "Angular traversability (nu)"),
    ]

    for ax, values, color, panel_title in panels:
        if values.size:
            counts, _, _ = ax.hist(
                values,
                bins=bins,
                density=False,
                log=True,
                color=color,
                alpha=0.75,
                edgecolor="white",
            )
            positive_counts = counts[counts > 0]
            if positive_counts.size:
                ymin = float(np.min(positive_counts)) * 0.8
                ymax = float(np.max(positive_counts)) * 1.1
                add_mean_median_vlines(ax, values, ymin, ymax)
                ax.legend(loc="best")
        else:
            ax.text(
                0.5,
                0.5,
                "No values available",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )

        ax.set_title(panel_title)
        ax.set_ylabel("Count (log scale)")
        ax.set_xlim(0.0, 1.0)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Traversability coefficient")
    fig.suptitle(
        t="Log-scale traversability distributions for high-confidence windows"
        f"(Confidence >= {confidence_threshold:.2f})",
        y=1.02,
    )
    fig.tight_layout()
    save_figure(fig, filename)


def plot_mission_confidence_distributions(stats_list, filename: str):
    if not stats_list:
        print(f"Skipping {filename}: no mission statistics available.")
        return

    bins = np.linspace(0.0, 1.0, 41)
    n_missions = len(stats_list)
    fig, axes = plt.subplots(
        n_missions,
        1,
        figsize=(12, max(3, n_missions * 1.2)),
        sharex=True,
        sharey=False,
    )

    if n_missions == 1:
        axes = np.array([axes])

    for row_idx, stats in enumerate(stats_list):
        name = stats["mission_short_name"]
        mu = np.asarray(stats.get("mu_confidences", np.array([])))
        nu = np.asarray(stats.get("nu_confidences", np.array([])))
        ax = axes[row_idx]

        if mu.size == 0 and nu.size == 0:
            ax.text(
                0.5,
                0.5,
                "No confidence data",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_ylabel(name)
            continue

        draw_above_below_on_axis(
            ax,
            mu,
            nu,
            bins,
            "tab:blue",
            "tab:orange",
            top_label="Linear confidence (mu)" if row_idx == 0 else None,
            bottom_label="Angular confidence (nu)" if row_idx == 0 else None,
            show_legend=row_idx == 0,
        )
        ax.set_ylabel(name)
        ax.set_xlim(0.0, 1.0)

    axes[-1].set_xlabel("Confidence")
    fig.supylabel("Density")
    fig.suptitle("Confidence Distributions per Mission", y=1.02)
    fig.tight_layout()
    save_figure(fig, filename)


def plot_simple_label_distributions(simple_stats_list: list, filename: str):
    """Plot tau_lin (top) vs tau_ang (bottom) distributions from simple baseline labels."""
    if not simple_stats_list:
        print(f"Skipping {filename}: no simple label statistics available.")
        return

    all_tau_lin = concatenate_non_empty([s["tau_lin"] for s in simple_stats_list])
    all_tau_ang = concatenate_non_empty([s["tau_ang"] for s in simple_stats_list])

    plot_above_below_distribution(
        all_tau_lin,
        all_tau_ang,
        filename=filename,
        title="Simple Baseline: Traversability Distributions Across the Entire Dataset",
        xlabel="Traversability score",
        top_label="Linear traversability (tau_lin)",
        bottom_label="Angular traversability (tau_ang)",
        top_color="tab:green",
        bottom_color="tab:purple",
    )


def plot_simple_label_per_mission(simple_stats_list: list, filename: str):
    """Per-mission traversability distributions for simple baseline labels."""
    if not simple_stats_list:
        print(f"Skipping {filename}: no simple label statistics available.")
        return

    bins = np.linspace(0.0, 1.0, 41)
    n_missions = len(simple_stats_list)
    fig, axes = plt.subplots(
        n_missions,
        1,
        figsize=(12, max(3, n_missions * 1.5)),
        sharex=True,
        sharey=False,
    )

    if n_missions == 1:
        axes = np.array([axes])

    for row_idx, stats in enumerate(simple_stats_list):
        name = stats["mission_short_name"]
        tau_lin = stats["tau_lin"]
        tau_ang = stats["tau_ang"]
        ax = axes[row_idx]

        draw_above_below_on_axis(
            ax,
            tau_lin,
            tau_ang,
            bins,
            "tab:green",
            "tab:purple",
            top_label="tau_lin" if row_idx == 0 else None,
            bottom_label="tau_ang" if row_idx == 0 else None,
            show_legend=row_idx == 0,
        )
        ax.set_ylabel(name)
        ax.set_xlim(0.0, 1.0)

    axes[-1].set_xlabel("Traversability score")
    fig.supylabel("Density")
    fig.suptitle("Simple Baseline: Traversability Distributions per Mission", y=1.02)
    fig.tight_layout()
    save_figure(fig, filename)


def plot_rhe_vs_simple_comparison(
    rhe_stats_list: list,
    simple_stats_list: list,
    filename: str,
):
    """Overlay RHE and simple baseline traversability distributions for comparison."""
    rhe_mu = concatenate_non_empty([s["mu_labels"] for s in rhe_stats_list])
    rhe_nu = concatenate_non_empty([s["nu_labels"] for s in rhe_stats_list])
    simple_lin = concatenate_non_empty([s["tau_lin"] for s in simple_stats_list])
    simple_ang = concatenate_non_empty([s["tau_ang"] for s in simple_stats_list])

    if not any(a.size for a in [rhe_mu, rhe_nu, simple_lin, simple_ang]):
        print(f"Skipping {filename}: no data available.")
        return

    bins = np.linspace(0.0, 1.0, 41)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

    for ax, rhe_vals, simple_vals, title in [
        (
            axes[0],
            rhe_mu,
            simple_lin,
            "Linear traversability: RHE mu vs simple tau_lin",
        ),
        (
            axes[1],
            rhe_nu,
            simple_ang,
            "Angular traversability: RHE nu vs simple tau_ang",
        ),
    ]:
        if rhe_vals.size:
            ax.hist(
                rhe_vals,
                bins=bins,
                density=True,
                alpha=0.55,
                label="RHE",
                color="tab:blue",
                edgecolor="white",
            )
        if simple_vals.size:
            ax.hist(
                simple_vals,
                bins=bins,
                density=True,
                alpha=0.55,
                label="Simple baseline",
                color="tab:green",
                edgecolor="white",
            )
        ax.set_title(title)
        ax.set_xlabel("Traversability score")
        ax.set_ylabel("Density")
        ax.set_xlim(0.0, 1.0)
        ax.legend(loc="best")
        ax.grid(True, alpha=0.3)

    fig.suptitle("RHE vs Simple Baseline Traversability Comparison", y=1.02)
    fig.tight_layout()
    save_figure(fig, filename)


def plot_mission_confidence_split_violin(stats_list, filename: str):
    if not stats_list:
        print(f"Skipping {filename}: no mission statistics available.")
        return

    import matplotlib.patches as patches
    from matplotlib.lines import Line2D

    mission_names = [stats["mission_short_name"] for stats in stats_list]
    positions = np.arange(len(mission_names))

    fig, ax = plt.subplots(figsize=(12, max(4, len(mission_names) * 0.5)))
    width = 0.8

    for idx, stats in enumerate(stats_list):
        mu = np.asarray(stats.get("mu_confidences", np.array([])))
        nu = np.asarray(stats.get("nu_confidences", np.array([])))
        pos = positions[idx]

        if mu.size:
            parts_mu = ax.violinplot(
                [mu],
                positions=[pos],
                vert=False,
                widths=width,
                points=VIOLIN_POINTS,
                bw_method=VIOLIN_BW_METHOD,
                showmeans=False,
                showmedians=False,
                showextrema=False,
            )
            for body in parts_mu["bodies"]:
                body.set_facecolor("tab:blue")
                body.set_alpha(0.7)
                clip = patches.Rectangle(
                    (0, pos), 1.0, width / 2.0, transform=ax.transData
                )
                body.set_clip_path(clip)

        if nu.size:
            parts_nu = ax.violinplot(
                [nu],
                positions=[pos],
                vert=False,
                widths=width,
                points=VIOLIN_POINTS,
                bw_method=VIOLIN_BW_METHOD,
                showmeans=False,
                showmedians=False,
                showextrema=False,
            )
            for body in parts_nu["bodies"]:
                body.set_facecolor("tab:orange")
                body.set_alpha(0.7)
                clip = patches.Rectangle(
                    (0, pos - width / 2.0), 1.0, width / 2.0, transform=ax.transData
                )
                body.set_clip_path(clip)

    ax.set_yticks(positions)
    ax.set_yticklabels(mission_names)
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel("Confidence")
    ax.set_title("Confidence split by mission")
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend(
        handles=[
            Line2D([0], [0], color="tab:blue", lw=6, label="mu confidence, upper half"),
            Line2D(
                [0], [0], color="tab:orange", lw=6, label="nu confidence, lower half"
            ),
        ],
        loc="best",
    )
    save_figure(fig, filename)

In [ ]:
CONFIDENCE_THRESHOLD = 0.6

SPLIT_HIGH_CONFIDENCE_FILES = False
SPLIT_CONFIDENCE_FILES = False

LOW_TRAVERSABILITY_THRESHOLD = 0.6
LOW_TRAVERSABILITY_COUNT = 5
LOW_TRAVERSABILITY_CONFIDENCE_THRESHOLD = 0.6

# Set to False to skip all simple baseline label loading and visualization.
VISUALIZE_SIMPLE_LABELS = True

processed_statistics = []

discovered_missions = discover_processed_missions(labels_root)
print(f"Discovered {len(discovered_missions)} processed missions in {labels_root}")

for mission_short_name, rhe_dir, simple_dir in discovered_missions:
    stats = load_mission_statistics(rhe_dir, mission_short_name)
    if stats is not None:
        processed_statistics.append(stats)

all_high_conf_mu = concatenate_non_empty([
    stats["high_conf_mu"] for stats in processed_statistics
])
all_high_conf_nu = concatenate_non_empty([
    stats["high_conf_nu"] for stats in processed_statistics
])
all_confidences = concatenate_non_empty([
    stats["combined_confidences"] for stats in processed_statistics
])
all_mu_confidences = concatenate_non_empty([
    stats["mu_confidences"] for stats in processed_statistics
])
all_nu_confidences = concatenate_non_empty([
    stats["nu_confidences"] for stats in processed_statistics
])

plot_high_confidence_traversability_distributions(
    all_high_conf_mu,
    all_high_conf_nu,
    filename="traversability_high_confidence_distributions.pdf",
    split_into_files=SPLIT_HIGH_CONFIDENCE_FILES,
)

plot_linear_angular_confidence_distributions(
    all_mu_confidences,
    all_nu_confidences,
    filename="confidence_distributions_all_missions.pdf",
    split_into_files=SPLIT_CONFIDENCE_FILES,
)

plot_mission_confidence_distributions(
    processed_statistics,
    filename="confidence_distributions_by_mission.pdf",
)

plot_mission_traversability_distributions(
    processed_statistics,
    filename="traversability_distributions_low_coeffs.pdf",
    low_value_threshold=LOW_TRAVERSABILITY_THRESHOLD,
    min_count=LOW_TRAVERSABILITY_COUNT,
    confidence_threshold=LOW_TRAVERSABILITY_CONFIDENCE_THRESHOLD,
)

plot_high_confidence_traversability_log_distributions(
    all_high_conf_mu,
    all_high_conf_nu,
    filename="traversability_distributions_low_coeffs_log.pdf",
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

# plot_mission_confidence_split_violin(
#     processed_statistics,
#     filename="confidence_split_violin_by_mission.pdf",
# )

# ── Simple baseline labels ────────────────────────────────────────────────────
if VISUALIZE_SIMPLE_LABELS:
    simple_statistics = []
    for mission_short_name, rhe_dir, simple_dir in discovered_missions:
        simple_stats = load_simple_mission_statistics(simple_dir, mission_short_name)
        if simple_stats is not None:
            simple_statistics.append(simple_stats)

    print(f"Loaded simple baseline labels for {len(simple_statistics)} missions.")

    plot_simple_label_distributions(
        simple_statistics,
        filename="simple_traversability_distributions_all_missions.pdf",
    )

    plot_simple_label_per_mission(
        simple_statistics,
        filename="simple_traversability_distributions_by_mission.pdf",
    )

    plot_rhe_vs_simple_comparison(
        processed_statistics,
        simple_statistics,
        filename="rhe_vs_simple_comparison.pdf",
    )

In [ ]:
import pandas as pd
from config.paths import WAYFASTER_DATASET_ROOT


def parse_traversability_vec(text, expected_len=2):
    vals = np.fromstring(str(text).strip()[1:-1], sep=" ", dtype=np.float64)
    if vals.size < expected_len:
        vals = np.pad(vals, (0, expected_len - vals.size), constant_values=np.nan)
    return vals[:expected_len]


def load_wayfaster_mu_nu(dataset_root: Path):
    mu_chunks = []
    nu_chunks = []
    total_duration_seconds = 0.0

    for states_path in sorted(dataset_root.rglob("states.csv")):
        try:
            states_df = pd.read_csv(states_path)
        except Exception as exc:
            print(f"Skipping {states_path.parent} (csv read failed): {exc}")
            continue

        if "traversability" not in states_df.columns:
            print(f"Skipping {states_path.parent} (missing 'traversability' column).")
            continue

        trav = np.vstack(
            states_df["traversability"].map(parse_traversability_vec).to_numpy()
        )
        mu = trav[:, 0]
        nu = trav[:, 1]

        mu = mu[np.isfinite(mu)]
        nu = nu[np.isfinite(nu)]

        if mu.size:
            mu_chunks.append(mu)
        if nu.size:
            nu_chunks.append(nu)

        # Calculate duration for this bag
        if "timestamp" in states_df.columns:
            ts = states_df["timestamp"].to_numpy(dtype=np.float64)
            ts_valid = ts[np.isfinite(ts)]
            if ts_valid.size > 1:
                bag_duration = float(np.max(ts_valid) - np.min(ts_valid))
                total_duration_seconds += bag_duration

    if not mu_chunks and not nu_chunks:
        raise RuntimeError(f"No valid traversability values found under {dataset_root}")

    wayfaster_mu = np.concatenate(mu_chunks) if mu_chunks else np.array([])
    wayfaster_nu = np.concatenate(nu_chunks) if nu_chunks else np.array([])

    return wayfaster_mu, wayfaster_nu, total_duration_seconds


def load_grandtour_mu_nu(stats_list):
    grandtour_mu = concatenate_non_empty([stats["mu_labels"] for stats in stats_list])
    grandtour_nu = concatenate_non_empty([stats["nu_labels"] for stats in stats_list])
    return grandtour_mu, grandtour_nu


def load_grandtour_durations(rhe_dirs):
    total_duration_seconds = 0.0
    for rhe_dir in rhe_dirs:
        timestamp_path = rhe_dir / "rhe_timestamp.npy"
        if timestamp_path.exists():
            ts = np.load(timestamp_path)
            ts_valid = ts[np.isfinite(ts)]
            if ts_valid.size > 1:
                mission_duration = float(np.max(ts_valid) - np.min(ts_valid))
                total_duration_seconds += mission_duration

    return total_duration_seconds


wayfaster_mu, wayfaster_nu, wayfaster_duration = load_wayfaster_mu_nu(
    WAYFASTER_DATASET_ROOT
)
grandtour_mu, grandtour_nu = load_grandtour_mu_nu(processed_statistics)
grandtour_duration = load_grandtour_durations([
    rhe_dir for _, rhe_dir, _ in discovered_missions
])

common_bins = np.linspace(0.0, 1.0, 51)

# Mu comparison histogram
fig_mu, ax_mu = plt.subplots(figsize=(11, 5.5))
ax_mu.hist(
    grandtour_mu,
    bins=common_bins,
    density=True,
    alpha=0.5,
    label=f"GrandTour ({grandtour_duration / 3600:.1f}h)",
    color="tab:blue",
    edgecolor="white",
)
ax_mu.hist(
    wayfaster_mu,
    bins=common_bins,
    density=True,
    alpha=0.5,
    label=f"WayFaster ({wayfaster_duration / 3600:.1f}h)",
    color="tab:red",
    edgecolor="white",
)
ax_mu.set_title("Mu Distribution Comparison: GrandTour vs WayFaster")
ax_mu.set_xlabel("mu")
ax_mu.set_ylabel("Density")
ax_mu.set_xlim(0.0, 1.0)
ax_mu.grid(True, alpha=0.3)
ax_mu.legend(loc="best")
fig_mu.tight_layout()
plt.show()

# Nu comparison histogram
fig_nu, ax_nu = plt.subplots(figsize=(11, 5.5))
ax_nu.hist(
    grandtour_nu,
    bins=common_bins,
    density=True,
    alpha=0.5,
    label=f"GrandTour ({grandtour_duration / 3600:.1f}h)",
    color="tab:blue",
    edgecolor="white",
)
ax_nu.hist(
    wayfaster_nu,
    bins=common_bins,
    density=True,
    alpha=0.5,
    label=f"WayFaster ({wayfaster_duration / 3600:.1f}h)",
    color="tab:red",
    edgecolor="white",
)
ax_nu.set_title("Nu Distribution Comparison: GrandTour vs WayFaster")
ax_nu.set_xlabel("nu")
ax_nu.set_ylabel("Density")
ax_nu.set_xlim(0.0, 1.0)
ax_nu.grid(True, alpha=0.3)
ax_nu.legend(loc="best")
fig_nu.tight_layout()
plt.show()

print(
    f"GrandTour: {grandtour_duration / 3600:.2f}h | WayFaster: {wayfaster_duration / 3600:.2f}h"
)